In [3]:
# Cell 1: 安装依赖
!pip install -qU langchain langchain-community chromadb sentence-transformers openai tiktoken

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.8.0 requires aiosqlite~=0.21.0, but you have aiosqlite 0.20.0 which is incompatible.
crewai 1.8.0 requires chromadb~=1.1.0, but you have chromadb 1.5.5 which is incompatible.
crewai 1.8.0 requires openai~=1.83.0, but you have openai 2.28.0 which is incompatible.
crewai 1.8.0 requires python-dotenv~=1.1.1, but you have python-dotenv 1.0.1 which is incompatible.
crewai 1.8.0 requires tokenizers~=0.20.3, but you have tokenizers 0.22.2 which is incompatible.
crewai-tools 1.8.0 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
instructor 1.12.0 requires openai<2.0.0,>=1.70.0, but you have openai 2.28.0 which is incompatible.
pandas 2.1.3 requires numpy<2,>=1.23.2; python_version == "3.11", but you have numpy 2.4.3 which is incompatible.


In [4]:
# Cell 2: 写入模拟简历并加载
import os
from langchain_community.document_loaders import TextLoader

# 1. 模拟写入一份 Markdown 格式的简历知识库
# resume_content = """
# # 个人信息
# - 姓名：张三
# - 职业：前端开发工程师 / AI 应用开发者
# - 经验：5年开发经验

# # 专业技能
# - 熟练掌握 HTML, CSS, JavaScript, React, Next.js。
# - 熟悉 Python, FastAPI，能够独立完成前后端全栈开发。
# - 深入了解 AI Agent 开发，熟练使用 LangChain 框架与提示词工程（Prompt Engineering）。
# - 熟练使用 Docker 进行容器化部署。

# # 项目经历
# ## 个人简历 AI 智能体 (交互式问答系统)
# - 使用技术：Next.js + FastAPI + LangChain + ChromaDB + DeepSeek LLM
# - 职责：全栈负责。利用 RAG 技术，将个人简历切块向量化，通过用户提问检索相关经历，并使用流式输出（SSE）给前端提供丝滑的打字机体验。
# """
# with open("resume_knowledge.md", "w", encoding="utf-8") as f:
#     f.write(resume_content)

# 2. 加载文档
loader = TextLoader("resume_knowledge.md", encoding="utf-8")
print(loader)
documents = loader.load()
print(f"成功加载文档，字符数: {len(documents[0].page_content)}")

D:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
# Cell 3: 文本切块
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 考虑到简历通常短小精悍，我们把 chunk_size 设置得小一点
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, 
    chunk_overlap=30,
    separators=["\n## ", "\n# ", "\n", "。", "！", "？", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"文档被切分成了 {len(chunks)} 个块 (Chunks)。")
print("第一块内容预览:", chunks[0].page_content)

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [ ]:
# Cell 4: 初始化 Embedding 与 ChromaDB
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma

# 使用 BAAI 的 bge-small-zh-v1.5 模型（轻量级，中文检索效果好）
model_name = "BAAI/bge-small-zh-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 构建 Chroma 向量库
persist_directory = "./chroma_db_resume"
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory=persist_directory
)

print("向量库创建成功并已持久化到本地！")

In [ ]:
# Cell 5: 向量检索与 System Prompt 拼接
user_query = "你能介绍一下张三在 AI Agent 方面的项目经验吗？"

# 1. 检索 Top-K (设 K=2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke(user_query)

# 2. 提取文本内容
context_text = "\n".join([doc.page_content for doc in retrieved_docs])

# 3. 拼装 System Prompt
system_prompt = f"""你是一个智能简历面试助手，你的任务是根据提供的简历知识库片段，准确、专业地回答用户的提问。
如果用户的提问在知识库片段中找不到答案，请直接回答“我的简历中暂未包含这部分信息”，不要自行编造。

【简历知识库片段】
{context_text}
"""

print("=== 检索到的上下文 ===")
print(context_text)
print("\n=== 构建的 System Prompt ===")
print(system_prompt)

In [ ]:
# Cell 6: DeepSeek API 调用与流式输出
from openai import OpenAI
import time
import sys

# 替换为你自己的 DeepSeek API Key
DEEPSEEK_API_KEY = "sk-xxxxxxxxxxxxxxxxxxxxxxxx" 

client = OpenAI(
    api_key=DEEPSEEK_API_KEY, 
    base_url="https://api.deepseek.com"
)

# 构建对话消息
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query}
]

print(">>> 正在连接 DeepSeek 大模型获取回答...\n")
print("AI 回答: ", end="")

# 开启流式请求
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    stream=True # 核心：开启流式输出
)

# 模拟 SSE 逐字打印输出到控制台
for chunk in response:
    content = chunk.choices[0].delta.content
    if content is not None:
        # sys.stdout.write 避免 print 自带的换行，实现打字机效果
        sys.stdout.write(content)
        sys.stdout.flush() 
        # 可以稍微加一点延迟让打字机效果更明显，但在实际生产 SSE 中不需要加 sleep
        time.sleep(0.02) 

print("\n\n>>> 输出完毕。")